In [29]:
# dependencies
import pandas as pd
import numpy as np
import zipfile
import requests
import io
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import math


# utils notebook

In [8]:
!rm -rf deep_learning_project

In [9]:
!git clone https://github.com/samarthsingh1/deep_learning_project

Cloning into 'deep_learning_project'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 116 (delta 29), reused 11 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 892.04 KiB | 6.03 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [10]:
!jupyter nbconvert --to script deep_learning_project/experimentation/data_utils.ipynb
%run deep_learning_project/experimentation/data_utils.py

[NbConvertApp] Converting notebook deep_learning_project/experimentation/data_utils.ipynb to script
[NbConvertApp] Writing 5212 bytes to deep_learning_project/experimentation/data_utils.py
(140256, 370)
False
41437378
              mean         max
MT_001    3.970785   48.223350
MT_002   20.768480  115.220484
MT_003    2.918308  151.172893
MT_004   82.184490  321.138211
MT_005   37.240309  150.000000
MT_006  141.227385  535.714286
MT_007    4.521338   44.657999
MT_008  191.401476  552.188552
MT_009   39.975354  157.342657
MT_010   42.205152  198.924731
                        total_load  hour_of_day  day_of_week  is_weekend  \
datetime                                                                   
2011-01-01 00:00:00  207058.270272            0            5           1   
2011-01-01 01:00:00  265378.510747            1            5           1   
2011-01-01 02:00:00  263924.219533            2            5           1   
2011-01-01 03:00:00  266306.134264            3            5 

/content/deep_learning_project/experimentation/data_utils.py:65: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.isna().sum()[0]==1
/content/deep_learning_project/experimentation/data_utils.py:73: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


In [11]:


train=train_scaled
val=val_scaled
test=test_scaled
scaler=scaler

create sequences

In [19]:
def create_sequences(data, input_window=168, forecast_horizon=24):
    feature_cols = ['total_load', 'hour_sin', 'hour_cos',
                    'dayofweek_sin', 'dayofweek_cos',
                    'month_sin', 'month_cos']

    features = data[feature_cols].values
    targets = data['total_load'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i : i + input_window])
        y.append(targets[i + input_window : i + input_window + forecast_horizon])

    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

In [20]:
X_train, y_train = create_sequences(train)
X_val, y_val = create_sequences(val)
X_test, y_test = create_sequences(test)

# print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
# print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
# print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: torch.Size([24354, 168, 7]), y_train: torch.Size([24354, 24])
X_val:   torch.Size([5068, 168, 7]),   y_val:   torch.Size([5068, 24])
X_test:  torch.Size([5070, 168, 7]),  y_test:  torch.Size([5070, 24])


Data loader ( batching)

In [21]:


def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=64):
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    # Sanity check
    x_batch, y_batch = next(iter(train_loader))
    print(f"Input batch shape:  {x_batch.shape}")
    print(f"Target batch shape: {y_batch.shape}")

    return train_loader, val_loader, test_loader



In [22]:
train_loader, val_loader, test_loader = create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test)

Input batch shape:  torch.Size([64, 168, 7])
Target batch shape: torch.Size([64, 24])


# Transformer

In [24]:


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: [1, max_len, d_model]

        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return x

In [25]:
# DELETE or skip any previous cell that defines ElectricityTransformer

class ElectricityTransformer(nn.Module):
    def __init__(self, input_features=7, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1, input_window=168,
                 forecast_horizon=24, num_quantiles=3):
        super().__init__()
        self.forecast_horizon = forecast_horizon
        self.num_quantiles = num_quantiles

        self.input_projection = nn.Linear(input_features, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=input_window)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_head = nn.Sequential(
            nn.Linear(d_model * input_window, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, forecast_horizon * num_quantiles)
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.reshape(x.size(0), -1)
        x = self.output_head(x)
        x = x.reshape(-1, self.forecast_horizon, self.num_quantiles)
        return x



In [26]:
class QuantileLoss(nn.Module):
    def __init__(self, quantiles=[0.1, 0.5, 0.9]):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, predictions, targets):
        # predictions shape: [batch, 24, num_quantiles]
        # targets shape: [batch, 24]

        targets = targets.unsqueeze(-1)  # [batch, 24, 1] for broadcasting
        losses = []

        for i, q in enumerate(self.quantiles):
            errors = targets - predictions[:, :, i].unsqueeze(-1)
            loss = torch.max(q * errors, (q - 1) * errors)
            losses.append(loss.mean())

        return sum(losses) / len(losses)

In [27]:
def train_model(model, train_loader, val_loader, criterion, num_epochs=30, lr=1e-3, device='cpu'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        epoch_train_loss = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(x_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- Validation ---
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                predictions = model(x_batch)
                loss = criterion(predictions, y_batch)
                epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # Update learning rate based on val loss
        scheduler.step(avg_val_loss)

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}")

    # Load best model
    model.load_state_dict(torch.load('best_model.pt'))
    print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

    return model, train_losses, val_losses

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = ElectricityTransformer().to(device)
criterion = QuantileLoss(quantiles=[0.1, 0.5, 0.9])

model, train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, num_epochs=30, device=device)

Using device: cuda
Epoch 1/30 | Train Loss: 0.0807 | Val Loss: 0.0437 | LR: 0.001000
Epoch 2/30 | Train Loss: 0.0487 | Val Loss: 0.0340 | LR: 0.001000
Epoch 3/30 | Train Loss: 0.0453 | Val Loss: 0.0329 | LR: 0.001000
Epoch 4/30 | Train Loss: 0.0443 | Val Loss: 0.0321 | LR: 0.001000
Epoch 5/30 | Train Loss: 0.0431 | Val Loss: 0.0329 | LR: 0.001000
Epoch 6/30 | Train Loss: 0.0420 | Val Loss: 0.0299 | LR: 0.001000
Epoch 7/30 | Train Loss: 0.0414 | Val Loss: 0.0307 | LR: 0.001000
Epoch 8/30 | Train Loss: 0.0412 | Val Loss: 0.0298 | LR: 0.001000
Epoch 9/30 | Train Loss: 0.0404 | Val Loss: 0.0308 | LR: 0.001000
Epoch 10/30 | Train Loss: 0.0400 | Val Loss: 0.0304 | LR: 0.001000
Epoch 11/30 | Train Loss: 0.0400 | Val Loss: 0.0346 | LR: 0.001000
Epoch 12/30 | Train Loss: 0.0393 | Val Loss: 0.0278 | LR: 0.001000
Epoch 13/30 | Train Loss: 0.0392 | Val Loss: 0.0297 | LR: 0.001000
Epoch 14/30 | Train Loss: 0.0386 | Val Loss: 0.0275 | LR: 0.001000
Epoch 15/30 | Train Loss: 0.0385 | Val Loss: 0.0284 